# Benchmark 2.1 — Part B (Kaggle)
## EfficientNetV2S · ConvNeXtTiny · ResNet50 · MobileViT-XS

**Jalankan notebook ini di Kaggle dengan GPU T4 / P100 (kuota gratis 30 jam/minggu).**
Setelah selesai, unduh file output dari tab **Output** lalu unggah ke notebook **Merge (Part C)**.

| Model | Paradigma / Catatan |
|---|---|
| **EfficientNetV2S** | Progressive Learning & Fused-MBConv |
| **ConvNeXtTiny** | Pure CNN berprinsip Swin Transformer (Modern) |
| **ResNet50** | Baseline Klasik Residual Learning |
| **MobileViT-XS** 🌟 | Hybrid Edge Vision Transformer (Global Attention) |

**Konfigurasi Dataset:**
- Dataset: [Roboflow `valll-rtclj/sapi-lokal/2`](https://universe.roboflow.com/valll-rtclj/sapi-lokal/dataset/2)
- 7 Kelas: `aceh, bali, limusin, madura, non_sapi, pasundan, po`
- Mitigasi Imbalance: **Balanced Class Weighting** & **Stratified 5-Fold Cross-Validation**

⏱ Estimasi Waktu Eksekusi: **~70–80 menit** di GPU T4/P100


In [ ]:
# ========================================
# 1. Setup Lingkungan & Verifikasi GPU
# ========================================
!pip install -q roboflow transformers scikit-learn seaborn matplotlib pandas

import os, gc, time, math, warnings, pickle, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from pathlib import Path
from collections import Counter

from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical, load_img, img_to_array

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 150
sns.set_style("whitegrid")

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✅ GPU Terdeteksi: {gpus[0].name}")
else:
    print("⚠️ GPU tidak terdeteksi — pastikan accelerator GPU T4/P100 aktif di sidebar Settings Kaggle.")
print(f"TensorFlow Version: {tf.__version__}")


In [ ]:
# ========================================
# 2. Unduh & Ekstraksi Dataset Sapi Lokal v2
# ========================================
from roboflow import Roboflow

ROBOFLOW_API_KEY = "oSqbdWgbnJkiVfOWl18o"
WORKSPACE_NAME   = "valll-rtclj"
PROJECT_NAME     = "sapi-lokal"
DATASET_VERSION  = 2

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE_NAME).project(PROJECT_NAME)
dataset = project.version(DATASET_VERSION).download("folder")
DATASET_DIR = dataset.location

IMAGE_SIZE = (224, 224)

raw_paths, raw_labels = [], []
for split in ["train", "valid", "test"]:
    split_dir = os.path.join(DATASET_DIR, split)
    if not os.path.exists(split_dir):
        continue
    for cls in sorted(os.listdir(split_dir)):
        cls_dir = os.path.join(split_dir, cls)
        if not os.path.isdir(cls_dir):
            continue
        for img in sorted(os.listdir(cls_dir)):
            if img.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
                raw_paths.append(os.path.join(cls_dir, img))
                raw_labels.append(cls)

print(f"Total citra terunduh: {len(raw_paths)}")

DEDUPLICATE_TO_RAW = True
if DEDUPLICATE_TO_RAW and len(raw_paths) > 350:
    base_map = {}
    for p, l in zip(raw_paths, raw_labels):
        fn = os.path.basename(p)
        base_prefix = fn.split(".rf.")[0] if ".rf." in fn else fn
        if base_prefix not in base_map:
            base_map[base_prefix] = (p, l)
    all_paths = [v[0] for v in base_map.values()]
    all_labels = [v[1] for v in base_map.values()]
    print(f"✅ Deduplikasi aktif: Menggunakan {len(all_paths)} citra asli mentah (un-augmented pool).")
else:
    all_paths = raw_paths
    all_labels = raw_labels
    print(f"Menggunakan {len(all_paths)} citra dari pool Roboflow.")

class_names = sorted(set(all_labels))
NUM_CLASSES  = len(class_names)
label_counts = Counter(all_labels)

print(f"\nKelas terdeteksi ({NUM_CLASSES}): {class_names}")
for c in class_names:
    print(f"  - {c:10s}: {label_counts[c]:3d} citra ({label_counts[c]/len(all_labels)*100:5.2f}%)")

print("\nMemuat citra ke memori array...")
X_all = np.zeros((len(all_paths), *IMAGE_SIZE, 3), dtype=np.float32)
for i, p in enumerate(all_paths):
    X_all[i] = img_to_array(load_img(p, target_size=IMAGE_SIZE))

label_to_idx = {n: i for i, n in enumerate(class_names)}
y_all     = np.array([label_to_idx[l] for l in all_labels])
y_all_cat = to_categorical(y_all, NUM_CLASSES)
print(f"Array masukan siap: X={X_all.shape}, y={y_all.shape}")


In [ ]:
# ========================================
# 3. Konfigurasi Eksperimen & Model Builder
# ========================================
BATCH_SIZE         = 16
PHASE1_EPOCHS      = 30
PHASE2_EPOCHS      = 20
PHASE1_LR          = 1e-3
PHASE2_LR          = 1e-5
N_FOLDS            = 5
UNFREEZE_FRACTION  = 0.3
RANDOM_STATE       = 42

OUTPUT_DIR = Path("/kaggle/working/benchmark2_1_outputs_partB") if os.path.exists("/kaggle") else Path("/content/benchmark2_1_outputs_partB")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def build_model(base_fn, preprocess_fn, num_classes, input_shape=(224, 224, 3), name="model"):
    if name == "MobileViT-XS":
        # Menggunakan TFMobileViT dari Hugging Face dengan bobot ImageNet pra-latih
        from transformers import TFMobileViTForImageClassification
        m = TFMobileViTForImageClassification.from_pretrained(
            "apple/mobilevit-xs",
            num_labels=num_classes,
            ignore_mismatched_sizes=True,
            id2label={i: c for i, c in enumerate(class_names)},
            label2id={c: i for i, c in enumerate(class_names)},
        )
        # Freeze backbone
        m.mobilevit.trainable = False
        m.compile(optimizer=tf.keras.optimizers.Adam(PHASE1_LR),
                  loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
                  metrics=["accuracy"])
        return m

    base = base_fn(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False
    x   = GlobalAveragePooling2D(name="gap")(base.output)
    x   = Dense(256, activation="relu", name="dense_head")(x)
    x   = Dropout(0.3, name="dropout_head")(x)
    out = Dense(num_classes, activation="softmax", name="output")(x)
    m   = Model(inputs=base.input, outputs=out, name=name)
    m.compile(optimizer=tf.keras.optimizers.Adam(PHASE1_LR),
              loss="categorical_crossentropy", metrics=["accuracy"])
    return m

def unfreeze_top(model, fraction=0.3, name="model"):
    if name == "MobileViT-XS":
        model.mobilevit.trainable = True
        model.compile(optimizer=tf.keras.optimizers.Adam(PHASE2_LR),
                      loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
                      metrics=["accuracy"])
        return

    base_layers = []
    for layer in model.layers:
        if layer.name == "gap":
            break
        base_layers.append(layer)
    for layer in base_layers:
        layer.trainable = False
    n = max(1, int(len(base_layers) * fraction))
    for layer in base_layers[-n:]:
        if not isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = True
    for layer in model.layers:
        if layer.name in ("dense_head", "dropout_head", "output"):
            layer.trainable = True
    model.compile(optimizer=tf.keras.optimizers.Adam(PHASE2_LR),
                  loss="categorical_crossentropy", metrics=["accuracy"])

def get_callbacks():
    return [
        EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, min_lr=1e-7, verbose=0),
    ]

def eval_model(model, X_val, y_val, preprocess_fn, name="model"):
    X_p = preprocess_fn(X_val.copy())
    if name == "MobileViT-XS":
        logits = model(X_p, training=False).logits
        probs  = tf.nn.softmax(logits, axis=-1).numpy()
    else:
        probs = model.predict(X_p, verbose=0)
    yp = np.argmax(probs, axis=1)
    return {
        "accuracy" : accuracy_score(y_val, yp),
        "precision": precision_score(y_val, yp, average="macro", zero_division=0),
        "recall"   : recall_score(y_val, yp, average="macro", zero_division=0),
        "f1"       : f1_score(y_val, yp, average="macro", zero_division=0),
        "y_pred"   : yp,
        "probs"    : probs,
        "pc_p"     : precision_score(y_val, yp, average=None, zero_division=0),
        "pc_r"     : recall_score(y_val, yp, average=None, zero_division=0),
        "pc_f1"    : f1_score(y_val, yp, average=None, zero_division=0),
    }

print("✅ Model builder & helper siap.")


In [ ]:
# ========================================
# 4. Training: 5-Fold CV × 4 Model (Part B) × 2 Phase
# ========================================
RESUME = False

MODEL_CONFIGS = {
    "EfficientNetV2S": {
        "base_fn"      : tf.keras.applications.EfficientNetV2S,
        "preprocess_fn": tf.keras.applications.efficientnet_v2.preprocess_input,
    },
    "ConvNeXtTiny": {
        "base_fn"      : tf.keras.applications.ConvNeXtTiny,
        "preprocess_fn": tf.keras.applications.convnext.preprocess_input,
    },
    "ResNet50": {
        "base_fn"      : tf.keras.applications.ResNet50,
        "preprocess_fn": tf.keras.applications.resnet50.preprocess_input,
    },
    "MobileViT-XS": {
        "base_fn"      : None,
        "preprocess_fn": lambda x: x / 255.0,
    },
}

print(f"Arsitektur Part B ({len(MODEL_CONFIGS)}): {list(MODEL_CONFIGS.keys())}")

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
ckpt = OUTPUT_DIR / "checkpoint_results_partB.pkl"

if RESUME and ckpt.exists():
    with open(ckpt, "rb") as f:
        all_results = pickle.load(f)
    completed = set(all_results.keys())
    print(f"Resume aktif — sudah selesai: {completed}")
else:
    all_results = {}
    completed   = set()

for model_name, cfg in MODEL_CONFIGS.items():
    if model_name in completed:
        print(f"SKIP {model_name} (sudah selesai)")
        continue

    print(f"\n{'='*70}")
    print(f"  MODEL: {model_name}")
    print(f"{'='*70}")

    mr = {
        "phase1": {"acc": [], "prec": [], "rec": [], "f1": [],
                   "y_trues": [], "y_preds": [], "probs": [],
                   "pc_p": [], "pc_r": [], "pc_f1": []},
        "phase2": {"acc": [], "prec": [], "rec": [], "f1": [],
                   "y_trues": [], "y_preds": [], "probs": [],
                   "pc_p": [], "pc_r": [], "pc_f1": []},
        "times": [], "class_names": class_names,
    }

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_all, y_all)):
        print(f"  Fold {fold+1}/{N_FOLDS}", end=" ")

        X_tr, X_va = X_all[tr_idx].copy(), X_all[va_idx].copy()
        y_tr_c     = y_all_cat[tr_idx]
        y_va_c     = y_all_cat[va_idx]
        y_tr_l     = y_all[tr_idx]
        y_va_l     = y_all[va_idx]

        classes_unique = np.unique(y_tr_l)
        cw_arr = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=y_tr_l)
        cw_dict = {i: float(w) for i, w in enumerate(cw_arr)}

        aug = ImageDataGenerator(
            preprocessing_function=cfg["preprocess_fn"],
            rotation_range=25, width_shift_range=0.1, height_shift_range=0.1,
            shear_range=0.1, zoom_range=0.2, horizontal_flip=True,
            brightness_range=[0.8, 1.2], fill_mode="nearest",
        )
        val_gen = ImageDataGenerator(preprocessing_function=cfg["preprocess_fn"])
        steps_tr = math.ceil(len(X_tr) / BATCH_SIZE)
        steps_va = math.ceil(len(X_va) / BATCH_SIZE)

        model = build_model(cfg["base_fn"], cfg["preprocess_fn"],
                            NUM_CLASSES, name=model_name)
        t0 = time.time()

        # Phase 1
        model.fit(
            aug.flow(X_tr, y_tr_c, batch_size=BATCH_SIZE, seed=RANDOM_STATE + fold),
            steps_per_epoch=steps_tr, epochs=PHASE1_EPOCHS,
            validation_data=val_gen.flow(X_va, y_va_c, batch_size=BATCH_SIZE, shuffle=False),
            validation_steps=steps_va, callbacks=get_callbacks(),
            class_weight=cw_dict, verbose=0,
        )
        m1 = eval_model(model, X_va, y_va_l, cfg["preprocess_fn"], name=model_name)
        for k in ["acc", "prec", "rec", "f1"]:
            mr["phase1"][k].append(m1[{"acc":"accuracy","prec":"precision","rec":"recall","f1":"f1"}[k]])
        mr["phase1"]["y_trues"].append(y_va_l)
        mr["phase1"]["y_preds"].append(m1["y_pred"])
        mr["phase1"]["probs"].append(m1["probs"])
        mr["phase1"]["pc_p"].append(m1["pc_p"])
        mr["phase1"]["pc_r"].append(m1["pc_r"])
        mr["phase1"]["pc_f1"].append(m1["pc_f1"])
        print(f"| P1 F1={m1['f1']:.3f} Acc={m1['accuracy']:.3f}", end=" ")

        # Phase 2
        unfreeze_top(model, UNFREEZE_FRACTION, name=model_name)
        model.fit(
            aug.flow(X_tr, y_tr_c, batch_size=BATCH_SIZE, seed=RANDOM_STATE + fold + 100),
            steps_per_epoch=steps_tr, epochs=PHASE2_EPOCHS,
            validation_data=val_gen.flow(X_va, y_va_c, batch_size=BATCH_SIZE, shuffle=False),
            validation_steps=steps_va, callbacks=get_callbacks(),
            class_weight=cw_dict, verbose=0,
        )
        elapsed = time.time() - t0
        mr["times"].append(elapsed)
        m2 = eval_model(model, X_va, y_va_l, cfg["preprocess_fn"], name=model_name)
        for k in ["acc", "prec", "rec", "f1"]:
            mr["phase2"][k].append(m2[{"acc":"accuracy","prec":"precision","rec":"recall","f1":"f1"}[k]])
        mr["phase2"]["y_trues"].append(y_va_l)
        mr["phase2"]["y_preds"].append(m2["y_pred"])
        mr["phase2"]["probs"].append(m2["probs"])
        mr["phase2"]["pc_p"].append(m2["pc_p"])
        mr["phase2"]["pc_r"].append(m2["pc_r"])
        mr["phase2"]["pc_f1"].append(m2["pc_f1"])
        print(f"| P2 F1={m2['f1']:.3f} Acc={m2['accuracy']:.3f} | {elapsed:.0f}s")

        if m2["f1"] >= max(mr["phase2"]["f1"]):
            if model_name == "MobileViT-XS":
                model.save_pretrained(OUTPUT_DIR / f"best_{model_name}")
            else:
                model.save(OUTPUT_DIR / f"best_{model_name}.keras")

        del model, X_tr, X_va
        tf.keras.backend.clear_session()
        gc.collect()

    all_results[model_name] = mr
    p2_f1 = mr["phase2"]["f1"]
    p2_acc = mr["phase2"]["acc"]
    print(f"  >> {model_name} Final Phase 2: Macro F1={np.mean(p2_f1):.4f} +/- {np.std(p2_f1):.4f} | Acc={np.mean(p2_acc)*100:.2f}%")
    with open(ckpt, "wb") as f:
        pickle.dump(all_results, f)

print("\n✅ SEMUA MODEL PART B SELESAI!")


In [ ]:
# ========================================
# 5. Evaluasi Deployment TFLite (Part B)
# ========================================
deployment = {}

for name in MODEL_CONFIGS:
    bp_keras = OUTPUT_DIR / f"best_{name}.keras"
    bp_hf    = OUTPUT_DIR / f"best_{name}"
    
    if not bp_keras.exists() and not bp_hf.exists():
        print(f"  {name}: model tidak ditemukan, skip")
        continue

    print(f"  Converting {name} to TFLite...", end=" ")
    try:
        if name == "MobileViT-XS":
            from transformers import TFMobileViTForImageClassification
            model = TFMobileViTForImageClassification.from_pretrained(bp_hf)
            converter = tf.lite.TFLiteConverter.from_keras_model(model)
            converter.target_spec.supported_ops = [
                tf.lite.OpsSet.TFLITE_BUILTINS,
                tf.lite.OpsSet.SELECT_TF_OPS
            ]
        else:
            model = tf.keras.models.load_model(bp_keras)
            converter = tf.lite.TFLiteConverter.from_keras_model(model)

        tflite_bytes = converter.convert()
        tflite_path = OUTPUT_DIR / f"{name}.tflite"
        with open(tflite_path, "wb") as f:
            f.write(tflite_bytes)

        tflite_mb = os.path.getsize(tflite_path) / 1024**2
        keras_mb  = os.path.getsize(bp_keras) / 1024**2 if bp_keras.exists() else 0.0
        params    = model.count_params()

        interp = tf.lite.Interpreter(model_path=str(tflite_path))
        interp.allocate_tensors()
        inp = interp.get_input_details()
        dummy = np.random.rand(1, 224, 224, 3).astype(np.float32)
        lats = []
        for _ in range(110):
            t0 = time.time()
            interp.set_tensor(inp[0]["index"], dummy)
            interp.invoke()
            lats.append((time.time() - t0) * 1000)
        avg_lat = np.mean(lats[10:])

        deployment[name] = {
            "params_m"  : params / 1e6,
            "keras_mb"  : keras_mb,
            "tflite_mb" : tflite_mb,
            "latency_ms": avg_lat,
        }
        print(f"{params/1e6:.2f}M | TFLite {tflite_mb:.1f}MB | CPU Latency {avg_lat:.1f}ms")
    except Exception as e:
        print(f"Gagal profiling TFLite {name}: {e}")
        deployment[name] = {"params_m": 0, "keras_mb": 0, "tflite_mb": 0, "latency_ms": 0}

    del model
    tf.keras.backend.clear_session()
    gc.collect()

with open(OUTPUT_DIR / "all_results_partB.pkl", "wb") as f:
    pickle.dump(all_results, f)
with open(OUTPUT_DIR / "deployment_partB.pkl", "wb") as f:
    pickle.dump(deployment, f)

print("✅ Artefak Part B tersimpan: all_results_partB.pkl & deployment_partB.pkl")


In [ ]:
# ========================================
# 6. Arsipkan Hasil ke ZIP & Unduh
# ========================================
zip_path = shutil.make_archive("/kaggle/working/benchmark2_1_outputs_partB" if os.path.exists("/kaggle") else "/content/benchmark2_1_outputs_partB", "zip", str(OUTPUT_DIR))
print(f"📦 ZIP Terbuat: {zip_path}")
print("Silakan unduh ZIP dari panel Output Kaggle / Files Colab.")
